### Table of Contents

1. [Introduction](#1.-introduction)

2. [Data](#2.-data)

3. [Methodology](#3.-methodology)

4. [Data Pre-processing](#4.-data-pre-processing)

    4.1. [Raw Dataset Loading & Inspection](#4.1.-raw-dataset-loading--inspection)

    4.2. [Conversion of WDI Dataset to Wide-format Panel Data](#4.2.-conversion-of-wdi-dataset-to-wide-format-panel-data)

    4.3. [Panel Balance Check](#4.3.-panel-balance-check)

    4.4. [Data Integrity Checks & Cleanup](#4.4.-data-integrity-checks--cleanup)

### 1. Introduction

**Research Question**  
Do countries’ economic growth and structural change explain variation in carbon dioxide emissions per capita? Does an increase in GDP per capita result in higher level of emissions?

**Motivation**    
While rapid industrial expansion and urbanization have boosted the global economy, they have introduced significant sustainability risks. Furthermore, as emerging factors like Digital and AI waste pose unprecedented risks, finding the balance between growth and sustainability is central to today’s economic policy debate. The subject of this research links macroeconomic performance and environmental outcomes while associating structural shifts for explainability.

**Background and Existing Findings**  
Simon Kuznets researched the relationship between a nation’s growth and its environmental effect, popularly known as the Environmental Kuznets Curve (EKC). Numerous studies in Environmental Economics have established an inverted-U relationship between GDP and emissions. However, results can vary depending on the class of pollutants, time periods, and country groups. Recent studies suggest that structural variables and energy composition may alter the growth-emission relationship, hence the scope for an updated assessment.

### 2. Data

We are using a balanced panel from the World Bank’s World Development Indicators database for all available countries between 2000 and 2022. The dataset will combine measures of CO₂ emissions, GDP per capita, trade, industrial value added (as a percentage of GDP), energy use, shares of renewable and fossil-fuel energy, urban population, and population density.

[World Bank World Development Indicators (WDI) Panel Data 2000-2022](https://github.com/priyadarshanparida/kuznets-hypothesis-growth-vs-sustainability-buan6312/blob/main/data/panel_raw_wdi_2000_2022.csv)

### 3. Methodology

We hypothesize CO₂ emissions per capita to initially rise with income and later decline as economies diversify and adopt cleaner energy sources – a pattern consistent with the Environmental Kuznets Curve. The economic research will estimate a linear model including income and emissions to capture the EKC shape with the coefficient on GDP being our parameter of interest. Panel fixed-effects estimation will be used to control for unobserved, time-invariant country characteristics.

In [65]:
import pandas as pd

### 4. Data Pre-processing

##### 4.1 Raw Dataset Loading & Inspection

In [66]:
# Load dataset
filepath = "../data/panel_raw_wdi_2000_2022.csv"
wdi_panel_raw = pd.read_csv(filepath)

# Basic structure checks
print(f"Dataset Shape: {wdi_panel_raw.shape}    ")
print(f"Columns: {wdi_panel_raw.columns.tolist()}")
print(f"Sample Observations: {wdi_panel_raw.head()}")

Dataset Shape: (2394, 27)    
Columns: ['Country Name', 'Country Code', 'Series Name', 'Series Code', '2000 [YR2000]', '2001 [YR2001]', '2002 [YR2002]', '2003 [YR2003]', '2004 [YR2004]', '2005 [YR2005]', '2006 [YR2006]', '2007 [YR2007]', '2008 [YR2008]', '2009 [YR2009]', '2010 [YR2010]', '2011 [YR2011]', '2012 [YR2012]', '2013 [YR2013]', '2014 [YR2014]', '2015 [YR2015]', '2016 [YR2016]', '2017 [YR2017]', '2018 [YR2018]', '2019 [YR2019]', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]']
Sample Observations:   Country Name Country Code  \
0  Afghanistan          AFG   
1  Afghanistan          AFG   
2  Afghanistan          AFG   
3  Afghanistan          AFG   
4  Afghanistan          AFG   

                                         Series Name           Series Code  \
0  Carbon dioxide (CO2) emissions excluding LULUC...  EN.GHG.CO2.PC.CE.AR5   
1                 GDP per capita (constant 2015 US$)        NY.GDP.PCAP.KD   
2       Energy use (kg of oil equivalent per capita)     EG.USE.P

##### 4.2 Conversion of WDI Dataset to Wide-format Panel Data

In [67]:
# 4.2.1 Melt the year columns (2000–2022) into a single Year column
wdi_panel_long = wdi_panel_raw.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    var_name="Year",
    value_name="Value"
)

# Clean up the Year column to retain only the numeric year (e.g., "2000" from "2000 [YR2000]")
wdi_panel_long["Year"] = wdi_panel_long["Year"].str.extract(r"(\d{4})").astype(int)

# Convert Value to numeric
wdi_panel_long["Value"] = pd.to_numeric(wdi_panel_long["Value"], errors="coerce")

# Quick check of the structure
print("Shape after melting:", wdi_panel_long.shape)
print(f"Missingness Summary: {wdi_panel_long.isna().sum()}")

Shape after melting: (55062, 6)
Missingness Summary: Country Name       0
Country Code       0
Series Name        0
Series Code        0
Year               0
Value           5982
dtype: int64


In [68]:
# 4.2.2 Pivot indicators so each variable becomes its own column

# Pivot: each Series Code becomes a separate column
wdi_panel = wdi_panel_long.pivot_table(
    index=["Country Name", "Country Code", "Year"],
    columns="Series Code",
    values="Value"
).reset_index()

# Drop the pivot_table column index name
wdi_panel.columns.name = None

# Rename key indicators for clarity
rename_map = {
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Year": "year",
    "EN.GHG.CO2.PC.CE.AR5": "co2_per_capita",
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "EG.USE.PCAP.KG.OE": "energy_use_per_capita",
    "SP.URB.TOTL.IN.ZS": "urban_pct",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "EG.FEC.RNEW.ZS": "renewable_pct",
    "NV.IND.TOTL.ZS": "industry_value_added_pct_gdp",
    "EG.USE.COMM.FO.ZS": "fossil_fuel_pct",
    "EN.POP.DNST": "population_density"
}
wdi_panel.rename(columns=rename_map, inplace=True)

# Sort for clean panel structure
wdi_panel = wdi_panel.sort_values(["country_name", "year"]).reset_index(drop=True)

# Quick check
print("Shape after pivot:", wdi_panel.shape)
print(wdi_panel.head())

# Missingness summary after pivot
print("\nMissingness Summary (post-pivot):")
print(wdi_panel.isna().sum())

Shape after pivot: (6084, 12)
  country_name country_code  year  renewable_pct  fossil_fuel_pct  \
0  Afghanistan          AFG  2000           45.0              NaN   
1  Afghanistan          AFG  2001           45.6              NaN   
2  Afghanistan          AFG  2002           37.8              NaN   
3  Afghanistan          AFG  2003           36.7              NaN   
4  Afghanistan          AFG  2004           44.2              NaN   

   energy_use_per_capita  co2_per_capita  population_density  trade_pct_gdp  \
0                    NaN        0.050476           30.863847            NaN   
1                    NaN        0.046573           31.099929            NaN   
2                    NaN        0.044078           32.776961            NaN   
3                    NaN        0.044341           34.854344            NaN   
4                    NaN        0.037898           36.123230            NaN   

   industry_value_added_pct_gdp  gdp_per_capita  urban_pct  
0                  

##### 4.3 Panel Balance Check

In [69]:
# 4.3 Panel Balance Check

# 4.3.1 Basic counts
print(f"Unique Countries: {wdi_panel['country_name'].nunique()}")
print(f"Unique Years: {wdi_panel['year'].nunique()}")
print(f"Year Range: {wdi_panel['year'].min()} — {wdi_panel['year'].max()}")

# 4.3.2 Expected observations (if perfectly balanced)
expected_obs = (
    wdi_panel["country_name"].nunique() *
    wdi_panel["year"].nunique()
)
print(f"Expected Observations (if balanced): {expected_obs}")
print(f"Actual Observations: {wdi_panel.shape[0]}")

# 4.3.3 Observations per country
country_year_counts = (
    wdi_panel.groupby("country_name")["year"]
    .nunique()
    .sort_values()
)
print("\nCountries with the fewest years of data:")
print(country_year_counts.head(10))

print("\nCountries with the most years of data:")
print(country_year_counts.tail(10))

# 4.3.4 Identify missing years per country
all_years = set(wdi_panel["year"].unique())
missing_years_by_country = {
    country: sorted(all_years - set(years))
    for country, years in wdi_panel.groupby("country_name")["year"].unique().items()
}

# Show a sample
print("\nSample Missing Years for First 5 Countries:")
for country in list(missing_years_by_country.keys())[:5]:
    print(country, ":", missing_years_by_country[country])


Unique Countries: 265
Unique Years: 23
Year Range: 2000 — 2022
Expected Observations (if balanced): 6095
Actual Observations: 6084

Countries with the fewest years of data:
country_name
St. Martin (French part)    12
Afghanistan                 23
Mozambique                  23
Myanmar                     23
Namibia                     23
Nauru                       23
Nepal                       23
Netherlands                 23
New Caledonia               23
New Zealand                 23
Name: year, dtype: int64

Countries with the most years of data:
country_name
Ghana                                       23
Gibraltar                                   23
Greece                                      23
Greenland                                   23
Grenada                                     23
Guam                                        23
Guatemala                                   23
Guinea                                      23
Fragile and conflict affected situations    23
Zim

##### 4.4 Data Integrity Checks & Cleanup

In [70]:
# 4.4.1 Check for duplicate rows
duplicates = wdi_panel.duplicated(subset=["country_name", "year"], keep=False)
print(f"Number of duplicate country-year rows: {duplicates.sum()}")

# 4.4.2 Confirm uniqueness of country-year keys
unique_pairs = wdi_panel[["country_name", "year"]].drop_duplicates().shape[0]
print(f"Unique country-year pairs: {unique_pairs}")
print(f"Total rows: {wdi_panel.shape[0]}")

# 4.4.3 Negative values check
numeric_cols = [
    "co2_per_capita",
    "gdp_per_capita",
    "energy_use_per_capita",
    "population_density",
    "renewable_pct",
    "fossil_fuel_pct",
    "industry_value_added_pct_gdp",
    "trade_pct_gdp",
    "urban_pct"
]

negative_counts = (wdi_panel[numeric_cols] < 0).sum()
print("\nNegative values detected:")
print(negative_counts)

# 4.4.4 Check for unrealistic percentage values (>100)
pct_cols = [
    "renewable_pct",
    "fossil_fuel_pct",
    "industry_value_added_pct_gdp",
    "trade_pct_gdp",
    "urban_pct"
]

over_100_counts = (wdi_panel[pct_cols] > 100).sum()
print("\nPercentage columns with values > 100:")
print(over_100_counts)

# 4.4.5 Missingness summary (post-cleaning)
print(f"\nMissingness Summary (final pre-processing):\n{wdi_panel.isna().sum()}")
print(f"\nTotal missing cells: {wdi_panel.isna().sum().sum()}")

# Reorder columns for better readability
ordered_cols = [
    "country_name",
    "country_code",
    "year",
    "co2_per_capita",
    "gdp_per_capita",
    "energy_use_per_capita",
    "urban_pct",
    "trade_pct_gdp",
    "renewable_pct",
    "fossil_fuel_pct",
    "industry_value_added_pct_gdp",
    "population_density"
]

wdi_panel = wdi_panel[ordered_cols]
wdi_panel.columns

Number of duplicate country-year rows: 0
Unique country-year pairs: 6084
Total rows: 6084

Negative values detected:
co2_per_capita                  0
gdp_per_capita                  0
energy_use_per_capita           0
population_density              0
renewable_pct                   0
fossil_fuel_pct                 6
industry_value_added_pct_gdp    0
trade_pct_gdp                   0
urban_pct                       0
dtype: int64

Percentage columns with values > 100:
renewable_pct                      0
fossil_fuel_pct                    0
industry_value_added_pct_gdp       0
trade_pct_gdp                   1343
urban_pct                          0
dtype: int64

Missingness Summary (final pre-processing):
country_name                       0
country_code                       0
year                               0
renewable_pct                    374
fossil_fuel_pct                 1592
energy_use_per_capita           1519
co2_per_capita                   311
population_density     

Index(['country_name', 'country_code', 'year', 'co2_per_capita',
       'gdp_per_capita', 'energy_use_per_capita', 'urban_pct', 'trade_pct_gdp',
       'renewable_pct', 'fossil_fuel_pct', 'industry_value_added_pct_gdp',
       'population_density'],
      dtype='object')

In [71]:
# 4.4.6: Keep only codes that are true ISO-3 country codes recognized by WDI
print("Shape before filtering countries:", wdi_panel.shape)
print(f"Unique countries in the WDI dataset before filtering to valid ISO-3 codes: {wdi_panel['country_name'].nunique()}")
wdi_countries = wdi_panel[wdi_panel["country_code"].isin(
    pd.read_csv("../data/country-codes.csv")["ISO3166-1-Alpha-3"].dropna().unique()
)]
print("Shape after filtering countries:", wdi_countries.shape)
print(f"Unique countries in the WDI dataset after filtering to valid ISO-3 codes: {wdi_countries['country_name'].nunique()}")
# Countries dropped
print("Countries dropped during filtering:", set(wdi_panel["country_name"]) - set(wdi_countries["country_name"]))

Shape before filtering countries: (6084, 12)
Unique countries in the WDI dataset before filtering to valid ISO-3 codes: 265
Shape after filtering countries: (4934, 12)
Unique countries in the WDI dataset after filtering to valid ISO-3 codes: 215
Countries dropped during filtering: {'Sub-Saharan Africa', 'Central Europe and the Baltics', 'Pre-demographic dividend', 'Kosovo', 'Sub-Saharan Africa (IDA & IBRD countries)', 'Africa Eastern and Southern', 'IDA blend', 'European Union', 'Pacific island small states', 'Euro area', 'Latin America & Caribbean', 'Latin America & Caribbean (excluding high income)', 'Late-demographic dividend', 'Lower middle income', 'North America', 'Upper middle income', 'IDA & IBRD total', 'OECD members', 'Latin America & the Caribbean (IDA & IBRD countries)', 'Europe & Central Asia (IDA & IBRD countries)', 'IDA total', 'Middle income', 'Sub-Saharan Africa (excluding high income)', 'Africa Western and Central', 'Post-demographic dividend', 'South Asia', 'East Asi